<a href="https://colab.research.google.com/github/a-forty-two/EY-MarApr26/blob/main/08_langchain_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Before starting this notebook, please create NVIDIA API KEY at https://build.nvidia.com/explore/discover

<br>

# <font color="">LangChain Expression Language</font>

<br>

In the previous notebook, we introduced some of the services we'll be taking advantage of for our LLM applications, including some external LLM platforms and a locally-hosted front-end service. Both of these components incorporate LangChain, but it hasn't been spotlighted as a major focus yet. It is expected that you have some experience with LangChain and LLMs, but this notebook is intended to catch you up in preparation for the later sections!

This notebook is designed to guide you through the integration and application of LangChain, a leading orchestration library for Large Language Models (LLMs), with the AI Foundation Endpoints from last time. Whether you are a seasoned developer or new to LLMs, this course will enhance your understanding and skills in building sophisticated LLM applications.

<br>

### **Learning Objectives:**

- Learning how to leverage chains and runnables to orchestrate interesting LLM systems.  
- Getting familiar with using LLMs for external conversation and internal reasoning.
- Be able to start up and run a simple [**Gradio**](https://www.gradio.app/) interface inside your notebook.

<br>

### **Questions To Think About:**

- What kinds of utilities might be necessary to keep information flowing through the pipeline **(primer for next notebook)**.
- When you encounter the [**Gradio**](https://www.gradio.app/), consider where all you have seen this style of interface before. Some possible places may include [**HuggingFace Spaces**](https://huggingface.co/spaces)...
- Near the end of the section, you'll learn that you can pass around chains as routes and access them across environments via ports. What kinds of requirements should you advertise if you are trying to receive chains from other microservices?

<br>

### **Environment Setup:**

In [1]:
%pip install -q langchain langchain-nvidia-ai-endpoints gradio

import os
os.environ["NVIDIA_API_KEY"] = ""

## If you encounter a typing-extensions issue, restart your runtime and try again
from langchain_nvidia_ai_endpoints import ChatNVIDIA
ChatNVIDIA.get_available_models()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.6 MB/s eta 0:00:00


[Model(id='speakleash/bielik-11b-v2.3-instruct', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable=None, thinking_param_disable=None, base_model=None),
 Model(id='meta/llama-guard-4-12b', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable=None, thinking_param_disable=None, base_model=None),
 Model(id='mistralai/mixtral-8x7b-instruct-v0.1', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=['ai-mixtral-8x7b-instruct', 'playground_mixtral_8x7b', 'mixtral_8x7b'], supports_tools=False, supports_structured_output=False, supports_thinking=False, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable=None, thinking_param_disable=None, 

----

<br>

## **Part 1:** What Is LangChain?

LangChain is an popular LLM orchestration library to help set up systems that have one or more LLM components. The library is, for better or worse, extremely popular and changes rapidly based on new developments in the field, meaning that somebody can have a lot of experience in some parts of LangChain while having little-to-no familiarity with other parts (either because there are just so many different features or the area is new and the features have only recently been implemented).

This notebook will be using the **LangChain Expression Language (LCEL)** to ramp up from basic chain specification to more advanced dialog management practices, so hopefully the journey will be enjoyable and even seasoned LangChain developers might learn something new!

<!-- > <img style="max-width: 400px;" src="imgs/langchain-diagram.png" /> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/langchain-diagram.png" width=400px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1NS7dmLf5ql04o5CyPZnd1gnXXgO8-jbR" width=400px/> -->

----

<br>

## **Part 2:** Chains and Runnables

When exploring a new library, it's important to note what are the core systems of the library and how are they used.

In LangChain, the main building block *used to be* the classic **Chain**: a small module of functionality that does something specific and can be linked up with other chains to make a system. So for all intents and purposes, it is a "building-block system" abstraction where the building blocks are easy to create, have consistent methods (`invoke`, `generate`, `stream`, etc), and can be linked up to work together as a system. Some example legacy chains include `LLMChain`, `ConversationChain`, `TransformationChain`, `SequentialChain`, etc.

More recently, a new recommended specification has emerged that is significantly easier to work with and extremely compact, the **LangChain Expression Language (LCEL)**. This new format relies on a different kind of primitive - a **Runnable** - which is simply an object that wraps a function. Allow dictionaries to be implicitly converted to Runnables and let a **pipe |** operator create a Runnable that passes data from the left to the right (i.e. `fn1 | fn2` is a Runnable), and you have a simple way to specify complex logic!

Here are some very representative example Runnables, created via the `RunnableLambda` class:

In [2]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from functools import partial

################################################################################
## Very simple "take input and return it"
identity = RunnableLambda(lambda x: x)  ## Or RunnablePassthrough works

################################################################################
## Given an arbitrary function, you can make a runnable with it
def print_and_return(x, preface=""):
    print(f"{preface}{x}")
    return x

rprint0 = RunnableLambda(print_and_return)

################################################################################
## You can also pre-fill some of values using functools.partial
rprint1 = RunnableLambda(partial(print_and_return, preface="1: "))

################################################################################
## And you can use the same idea to make your own custom Runnable generator
def RPrint(preface=""):
    return RunnableLambda(partial(print_and_return, preface=preface))

################################################################################
## Chaining two runnables
chain1 = identity | rprint0
chain1.invoke("Hello World!")
print()
print('*****************************************')
print('*****************************************')
################################################################################
## Chaining that one in as well
output = (
    chain1           ## Prints "Welcome Home!" & passes "Welcome Home!" onward
    | rprint1        ## Prints "1: Welcome Home!" & passes "Welcome Home!" onward
    | RPrint("2: ")  ## Prints "2: Welcome Home!" & passes "Welcome Home!" onward
).invoke("Welcome Home!")
print('*****************************************')
print('*****************************************')
## Final Output Is Preserved As "Welcome Home!"
print("\nOutput:", output)

Hello World!

*****************************************
*****************************************
Welcome Home!
1: Welcome Home!
2: Welcome Home!
*****************************************
*****************************************

Output: Welcome Home!


----

<br>

## **Part 3:** Dictionary Pipelines with Chat Models

There's a lot you can do with runnables, but it's important to formalize some best practices. At the moment, it's easiest to use *dictionaries* as our default variable containers for a few key reasons:

**Passing dictionaries helps us keep track of our variables by name.**

Since dictionaries allow us to propagate named variables (values referenced by keys), using them is great for locking in our chain components' outputs and expectations.

**LangChain prompts expect dictionaries of values.**

It's quite intuitive to specify an LLM Chain in LCEL to take in a dictionary and produce a string, and equally easy to raise said string back up to be a dictionary. This is very intentional and is partially due to the above reason.

<br>

### **Example 1:** A Simple LLM Chain

One of the most fundamental components of classical LangChain is the `LLMChain` that accepts a **prompt** and an **LLM**:

- A prompt, usually retrieved from a call like `PromptTemplate.from_template("string with {key1} and {key2}")`, specifies a template for creating a string as output. A dictionary `{"key1" : 1, "key2" : 2}` could be passed in to get the output `"string with 1 and 2"`.
    - For chat models like `ChatNVIDIA`, you would use `ChatPromptTemplate.from_messages` instead.
- An LLM takes in a string and returns a generated string.
    - Chat models like `ChatNVIDIA` work with messages instead, but it's the same idea! Using an **StrOutputParser** at the end will extract the content from the message.

The following is a lightweight example of a simple chat chain as described above. All it does is take in an input dictionary and use it fill in a system message to specify the overall meta-objective and a user input to specify query the model.

In [5]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## Simple Chat Pipeline
chat_llm = ChatNVIDIA(model="meta/llama3-8b-instruct")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Only respond in rhymes"),
    ("user", "{input}")
])

rhyme_chain = prompt | chat_llm | StrOutputParser()

print(rhyme_chain.invoke({"input" : "Tell me about birds!"}))

Birds are quite a sight,
Feathered friends that take to flight.
They chirp and tweet with glee,
A symphony for you and me!


<br>

In addition to just using the code command as-is, we can try using a [**Gradio interface**](https://www.gradio.app/guides/creating-a-chatbot-fast) to play around with our model. Gradio is a popular tool that provides simple building blocks for creating custom generative AI interfaces! The below example shows how you can make an easy gradio chat interface with this particular example chain:

In [7]:
import gradio as gr

#######################################################
## Non-streaming Interface like that shown above

# def rhyme_chat(message, history):
#     return rhyme_chain.invoke({"input" : message})

# gr.ChatInterface(rhyme_chat).launch()

#######################################################
## Streaming Interface

def rhyme_chat_stream(message, history):
    ## This is a generator function, where each call will yield the next entry
    buffer = ""
    for token in rhyme_chain.stream({"input" : message}):
        buffer += token
        yield buffer

## Uncomment when you're ready to try this.
demo = gr.ChatInterface(rhyme_chat_stream).queue()
window_kwargs = {} # or {"server_name": "0.0.0.0", "root_path": "/7860/"}
demo.launch(share=True, debug=True, **window_kwargs)

## IMPORTANT!! When you're done, please click the Square button (twice to be safe) to stop the session.

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://847069c0e2ad46f952.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://847069c0e2ad46f952.gradio.live


<br>

### **Example 2: Internal Response**

Sometimes, you also want to have some quick reasoning that goes on behind the scenes before your response actually comes out to the user. When performing this task, you need a model with a strong instruction-following prior assumption built-in.

The following is an example "zero-shot classification" pipeline which will try to categorize a sentence into one of a couple of classes.

**In order, this zero-shot classification chain:**
- Takes in a dictionary with two required keys, `input` and `options`.
- Passes it through the zero-shot prompt to get the input to our LLM.
- Passes that string to the model to get the result.

**Task:** Pick out several models that you think would be good for this kind of task and see how well they perform! Specifically:
- **Try to find models that are predictable across multiple examples.** If the format is always easy to parse and extremely predictable, then the model is probably ok.
- **Try to find models that are also fast!** This is important because internal reasoning generally happens behind the hood before the external response gets generated. Thereby, it is a blocking process which can slow down start of "user-facing" generation, making your system feel sluggish.

In [10]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## Feel free to try out some more models and see if there are better lightweight options
## https://build.nvidia.com
instruct_llm = ChatNVIDIA(model="mistralai/mistral-7b-instruct-v0.2")

sys_msg = (
    "Choose the most likely topic classification given the sentence as context."
    " Only one word, no explanation.\n[Options : {options}]"
)

## One-shot classification prompt with heavy format assumptions.
zsc_prompt = ChatPromptTemplate.from_messages([
    ("system", sys_msg),
    ("user", "[[The sea is awesome]]"),
    ("assistant", "boat"),
    ("user", "[[This is not ripe enough]]"),
    ("assistant", "mango"),
    ("user", "[[{input}]]"),
])

## Roughly equivalent as above for <s>[INST]instruction[/INST]response</s> format
zsc_prompt = ChatPromptTemplate.from_template(
    f"{sys_msg}\n\n"
    "[[The sea is awesome]][/INST]boat</s><s>[INST]"
    "[[{input}]]"
)

zsc_chain = zsc_prompt | instruct_llm | StrOutputParser()

def zsc_call(input, options=["car", "boat", "airplane", "bike", "mango", "banana"]):
    return zsc_chain.invoke({"input" : input, "options" : options}).split()[0]

print("-" * 80)
print(zsc_call("Should I take the next exit, or keep going to the next one?"))

print("-" * 80)
print(zsc_call("this is too sweet, let's make it into pudding"))

print("-" * 80)
print(zsc_call("I'm scared of heights, so flying probably isn't for me"))

--------------------------------------------------------------------------------
car
--------------------------------------------------------------------------------
mango
--------------------------------------------------------------------------------
airplane


<br>

### **Example 3: Multi-Component Chains**

The previous example showed how we can coerce a dictionary into a string by passing it through a `prompt -> LLM` chain, so that's one easy structure to motivate the container choice. But is it just as easy to convert the string output back up to a dictionary?

**Yes, it is!** The simplest way is actually to use the LCEL *"implicit runnable"* syntax, which allows you to use a dictionary of functions (including chains) as a runnable that runs each function and maps the value to the key in the output dictionary.

The following is an example which exercises these utilities while also providing a few extra tools you may find useful in practice.

In [12]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from functools import partial

################################################################################
## Example of dictionary enforcement methods
def make_dictionary(v, key):
    if isinstance(v, dict):
        return v
    return {key : v}

def RInput(key='input'):
    '''Coercing method to mold a value (i.e. string) to in-like dict'''
    return RunnableLambda(partial(make_dictionary, key=key))

def ROutput(key='output'):
    '''Coercing method to mold a value (i.e. string) to out-like dict'''
    return RunnableLambda(partial(make_dictionary, key=key))

def RPrint(preface=""):
    return RunnableLambda(partial(print_and_return, preface=preface))

################################################################################
## Common LCEL utility for pulling values from dictionaries
from operator import itemgetter

up_and_down = (
    RPrint("A: ")
    ## Custom ensure-dictionary process
    | RInput()
    | RPrint("B: ")
    ## Pull-values-from-dictionary utility
    | itemgetter("input")
    | RPrint("C: ")
    ## Anything-in Dictionary-out implicit map
    | {
        'word1' : (lambda x : x.split()[0]),
        'word2' : (lambda x : x.split()[1]),
        'words' : (lambda x: x),  ## <- == to RunnablePassthrough()
    }
    | RPrint("D: ")
    | itemgetter("word1")
    | RPrint("E: ")
    ## Anything-in anything-out lambda application
    | RunnableLambda(lambda x: x.upper())
    | RPrint("F: ")
    ## Custom ensure-dictionary process
    | ROutput()
)

up_and_down.invoke({"input" : "Hello World"})

A: {'input': 'Hello World'}
B: {'input': 'Hello World'}
C: Hello World
D: {'word1': 'Hello', 'word2': 'World', 'words': 'Hello World'}
E: Hello
F: HELLO


{'output': 'HELLO'}

In [ ]:
## NOTE how the dictionary enforcement methods make it easy to make the following syntax equivalent
up_and_down.invoke("Hello World")